# Train the Rater — selectable outer/meta loss

This notebook trains only the Rater and saves a reusable `final_rater.pt`.
The final classifier remains a later experiment.

Representation and inner training:

- frozen Waterbirds-ERM ResNet-50 2048-D embeddings;
- two independent random-from-scratch `Linear(2048,2)` inner classifiers;
- every training example receives a Rater-derived weight;
- each inner classifier sees the full training split once per Rater update;
- 500 meta steps by default.

## Outer/meta-loss choices

Set `OUTER_LOSS` in cell 6 to one of:

- `standard_ce`: mean CE over all outer/meta examples.
- `worst_group_loss`: compute mean CE for each observed group in the outer/meta split, then take the largest group mean. This uses group annotations as a group-supervised meta-training ablation.
- `group_balanced_loss`: compute mean CE for each observed group, then average group means equally. This also uses group annotations.
- `classwise_top_loss`: average the top `OUTER_TOP_LOSS_PERCENT` CE losses within each class, then average the classes equally.
- `class_balanced`: average CE within each class, then average the class means equally.
- `worst_class_loss`: take the largest class mean CE.
- `logsumexp`: normalized log-sum-exp over all outer examples,
  `(logsumexp(beta * CE) - log(N)) / beta`. Larger `OUTER_LSE_BETA`
  emphasizes high-loss examples.

Group annotations are read from the outer/meta dataset only for the two group-based objectives. The inner Rater weighting rule is unchanged. Groups absent from the outer split are excluded from the group aggregation.


In [ ]:
# 1. Paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime
import os, sys, subprocess, shutil, re, torch

DRIVE_ROOT = Path("/content/drive/MyDrive/evidential_alignment_colab")
REPO_DIR = Path("/content/evidential_alignment")
DATA_ROOT = Path("/content/datasets")
WB_DIR = DATA_ROOT / "waterbird_complete95_forest2water2"
WB_ARCHIVE = DRIVE_ROOT / "archives" / "waterbird_complete95_forest2water2.tar.gz"

ERM_MODEL = DRIVE_ROOT / "erm_base_models" / "waterbirds_erm_resnet50_imagenet_100ep_best_val_acc.pt"
if not ERM_MODEL.exists():
    ERM_MODEL = DRIVE_ROOT / "erm_base_models" / "erm_waterbirds_resnet50_train_128B_100E_seed0_pretrained" / "best_val_acc_model.pt"
assert ERM_MODEL.exists(), ERM_MODEL

EMBED_ROOT = DRIVE_ROOT / "embeddings_waterbirds_erm"
OUTPUT_ROOT = DRIVE_ROOT / "logsumexp_runs"
LOCAL_PRE = Path("/content/rater_precompute_local")
LOCAL_RUN = Path("/content/rater_meta_only_local")

for p in [WB_ARCHIVE.parent, EMBED_ROOT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("ERM:", ERM_MODEL)
print("Embedding cache:", EMBED_ROOT)
print("Rater outputs:", OUTPUT_ROOT)

Mounted at /content/drive
ERM: /content/drive/MyDrive/evidential_alignment_colab/erm_base_models/waterbirds_erm_resnet50_imagenet_100ep_best_val_acc.pt
Embedding cache: /content/drive/MyDrive/evidential_alignment_colab/embeddings_waterbirds_erm
Rater outputs: /content/drive/MyDrive/evidential_alignment_colab/logsumexp_runs


In [ ]:
# 2. Fresh repo + dependencies
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone",
    "https://github.com/TheRealMmd/evidential_alignment.git",
    str(REPO_DIR),
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "timm", "transformers", "wilds", "wget", "matplotlib", "scipy",
], check=True)

print("✓ repo ready")

✓ repo ready


In [ ]:
# 3. Waterbirds
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not (WB_ARCHIVE.exists() and WB_ARCHIVE.stat().st_size > 100_000_000):
    subprocess.run([
        "wget", "-c",
        "https://downloads.cs.stanford.edu/nlp/data/dro/waterbird_complete95_forest2water2.tar.gz",
        "-O", str(WB_ARCHIVE),
    ], check=True)

if not (WB_DIR / "metadata.csv").exists():
    subprocess.run([
        "tar", "-xzf", str(WB_ARCHIVE), "-C", str(DATA_ROOT)
    ], check=True)

assert (WB_DIR / "metadata.csv").exists()
print("✓ Waterbirds ready")

✓ Waterbirds ready


In [ ]:
# 4. Patch data path + PyTorch 2.6 loading
config_path = REPO_DIR / "config.py"
text = config_path.read_text(encoding="utf-8")
placeholder = '"waterbirds": "/path/to/datasets/waterbirds"'
if placeholder in text:
    text = text.replace(placeholder, f'"waterbirds": "{WB_DIR}"')
else:
    text, n = re.subn(
        r'("waterbirds"\s*:\s*)["\'][^"\']*["\']',
        lambda m: m.group(1) + '"' + str(WB_DIR) + '"',
        text,
        count=1,
    )
    assert n == 1
config_path.write_text(text, encoding="utf-8")

algorithm_path = REPO_DIR / "algorithms" / "algorithm.py"
text = algorithm_path.read_text(encoding="utf-8")
text = text.replace(
    'torch.load(check_point, map_location="cpu")',
    'torch.load(check_point, map_location="cpu", weights_only=False)',
)
text = text.replace(
    "torch.load(check_point, map_location='cpu')",
    'torch.load(check_point, map_location="cpu", weights_only=False)',
)
algorithm_path.write_text(text, encoding="utf-8")
print("✓ repo patched")

✓ repo patched


In [ ]:
# 5. Install the updated Rater implementation
#
# Upload algorithms/rater.py from this project the first time, or whenever
# it changes. The notebook stores the uploaded source on Drive for reuse.

import ast
from google.colab import files

CODE_DIR = DRIVE_ROOT / "code"
CODE_DIR.mkdir(parents=True, exist_ok=True)
SAVED_RATER_SOURCE = CODE_DIR / "rater_classwise_top_loss.py"

REQUIRED_SOURCE_MARKERS = [
    "RATER_OUTER_LOSS",
    "_compute_outer_objective",
    "standard_ce",
    "worst_group_loss",
    "group_balanced_loss",
    "class_balanced",
    "worst_class_loss",
    "logsumexp",
]

def is_current_rater_source(source):
    return all(marker in source for marker in REQUIRED_SOURCE_MARKERS)

source_text = ""
if SAVED_RATER_SOURCE.exists():
    candidate_text = SAVED_RATER_SOURCE.read_text(encoding="utf-8")
    if is_current_rater_source(candidate_text):
        source_text = candidate_text
        print("Using current saved Rater source:", SAVED_RATER_SOURCE)

if not source_text:
    print("Upload algorithms/rater.py from this project.")
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.endswith(".py")]
    if len(candidates) != 1:
        raise RuntimeError("Upload exactly one Python file: algorithms/rater.py")
    source_text = uploaded[candidates[0]].decode("utf-8")
    if not is_current_rater_source(source_text):
        raise RuntimeError(
            "The uploaded rater.py does not include all outer-loss options."
        )
    SAVED_RATER_SOURCE.write_text(source_text, encoding="utf-8")
    print("Saved current Rater source to:", SAVED_RATER_SOURCE)

ast.parse(source_text)

rater_path = REPO_DIR / "algorithms" / "rater.py"
rater_path.write_text(source_text, encoding="utf-8")

init_path = REPO_DIR / "algorithms" / "__init__.py"
init_text = init_path.read_text(encoding="utf-8")
if "from . import rater" not in init_text:
    init_path.write_text(init_text.rstrip() + "\nfrom . import rater\n", encoding="utf-8")

print("Rater code with selectable outer losses installed.")


Upload algorithms/rater.py from this project.


Saving rater.py to rater.py
Saved current Rater source to: /content/drive/MyDrive/evidential_alignment_colab/code/rater_classwise_top_loss.py
Rater code with selectable outer losses installed.


In [ ]:
# 6. Experiment parameters
META_STEPS = 500
BATCH_SIZE = 128
NUM_INNER_MODELS = 2

# Choose: standard_ce, worst_group_loss, group_balanced_loss,
#         classwise_top_loss, class_balanced, worst_class_loss, logsumexp
OUTER_LOSS = "logsumexp"
OUTER_TOP_LOSS_PERCENT = 30.0  # used only for classwise_top_loss
OUTER_LSE_BETA = 1.0           # used only for logsumexp

assert OUTER_LOSS in {
    "standard_ce", "worst_group_loss", "group_balanced_loss",
    "classwise_top_loss", "class_balanced", "worst_class_loss", "logsumexp"
}
assert 0.0 < OUTER_TOP_LOSS_PERCENT <= 100.0
assert OUTER_LSE_BETA > 0.0

RATER_WEIGHTING = "sigmoid"
INNER_LR = 2e-4
TEMPERATURE = 1.0
OUTER_LR = 3e-4
GRAD_CLIP = 5.0

# Diagnostics: evaluate often, but avoid hundreds of PNGs.
EVAL_FREQ = 10
PLOT_FREQ = 25

os.environ.update({
    "RATER_ERM_MODEL": str(ERM_MODEL),
    "RATER_EMBEDDING_CACHE_ROOT": str(EMBED_ROOT),
    "RATER_META_STEPS": str(META_STEPS),
    "RATER_NUM_INNER_MODELS": str(NUM_INNER_MODELS),
    "RATER_WEIGHTING": RATER_WEIGHTING,
    "RATER_INNER_LR": str(INNER_LR),
    "RATER_OUTER_LR": str(OUTER_LR),
    "RATER_TEMPERATURE": str(TEMPERATURE),
    "RATER_GRAD_CLIP": str(GRAD_CLIP),
    "RATER_OUTER_LOSS": OUTER_LOSS,
    "RATER_OUTER_TOP_LOSS_PERCENT": str(OUTER_TOP_LOSS_PERCENT),
    "RATER_OUTER_LSE_BETA": str(OUTER_LSE_BETA),
    "RATER_EVAL_FREQ": str(EVAL_FREQ),
    "RATER_PLOT_FREQ": str(PLOT_FREQ),
})

print({
    "meta_steps": META_STEPS,
    "batch_size": BATCH_SIZE,
    "inner_models": NUM_INNER_MODELS,
    "outer_loss": OUTER_LOSS,
    "outer_top_loss_percent_per_class": OUTER_TOP_LOSS_PERCENT,
    "outer_lse_beta": OUTER_LSE_BETA,
    "outer_uses_group_annotations": OUTER_LOSS in {"worst_group_loss", "group_balanced_loss"},
})


In [ ]:
# 7. Precompute/reuse persistent ERM embeddings
os.environ["RATER_PRECOMPUTE_EMBEDDINGS_ONLY"] = "1"
shutil.rmtree(LOCAL_PRE, ignore_errors=True)
LOCAL_PRE.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "main.py",
    "--dataset", "waterbirds",
    "--algorithm", "rater",
    "--backbone", "resnet50",
    "--pretrained", "True",
    "--erm_model", str(ERM_MODEL),
    "--save_folder", str(LOCAL_PRE),
    "--train_split", "train",
    "--split_val", "0.4",
    "--mode", "train",
    "--epoch", str(META_STEPS),
]

result = subprocess.run(cmd, cwd=str(REPO_DIR), env=os.environ.copy())
if result.returncode != 0:
    raise RuntimeError("Embedding preparation failed")

shutil.rmtree(LOCAL_PRE, ignore_errors=True)
print("✓ persistent ERM embeddings ready")

✓ persistent ERM embeddings ready


In [ ]:
# 8. Verify cache
cache_files = sorted(EMBED_ROOT.rglob("*.pt"))
for p in cache_files:
    print(p.relative_to(EMBED_ROOT), f"{p.stat().st_size / 1024**2:.2f} MB")

assert any(p.name == "train_no_aug.pt" for p in cache_files)
assert any("val_subset1" in p.name for p in cache_files)
assert any("val_subset2" in p.name for p in cache_files)
assert any(p.name == "test.pt" for p in cache_files)

sample = next(p for p in cache_files if p.name == "train_no_aug.pt")
payload = torch.load(sample, map_location="cpu", weights_only=False)
assert payload["cache_metadata"]["representation_source"] == "waterbirds_erm_checkpoint"
print("✓ cache verified:", tuple(payload["embeddings"].shape))

waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/test.pt 45.40 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/train_no_aug.pt 37.57 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val.pt 9.40 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p4_seed_0.pt 3.76 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_0.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_1.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_11.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_42.pt 5.64 MB
waterbirds/resnet50_waterbirds_erm_1055b1e8de29/resolution_224/val_subset1_splittrain_1p0_splitval_0p6_seed_63.pt 5.64 MB
waterbirds/resnet50_water

In [ ]:
# 9. Run Rater meta-learning with the selected outer loss
os.environ["RATER_PRECOMPUTE_EMBEDDINGS_ONLY"] = "0"
shutil.rmtree(LOCAL_RUN, ignore_errors=True)
LOCAL_RUN.mkdir(parents=True, exist_ok=True)

# Rater hyperparameters live in environment variables, so the command is short.
cmd = [
    sys.executable, "-u", "main.py",
    "--dataset", "waterbirds",
    "--algorithm", "rater",
    "--backbone", "resnet50",
    "--pretrained", "True",
    "--erm_model", str(ERM_MODEL),
    "--save_folder", str(LOCAL_RUN),
    "--train_split", "train",
    "--test_split", "test",
    "--split_val", "0.4",
    "--batch_size", str(BATCH_SIZE),
    "--mode", "train",
    "--epoch", str(META_STEPS),
    "--seed", "0",
]

result = subprocess.run(cmd, cwd=str(REPO_DIR), env=os.environ.copy())
if result.returncode != 0:
    raise RuntimeError("Rater meta-training failed")

print("✓ Rater meta-training complete")

In [ ]:
# 10. Copy only model + plots + log + summary to Drive
rater_files = sorted(LOCAL_RUN.rglob("final_rater.pt"))
assert len(rater_files) == 1, f"found {len(rater_files)} final_rater.pt files"

local_exp = rater_files[0].parent
for name in ["final_rater.pt", "log.txt", "summary.txt", "plots"]:
    assert (local_exp / name).exists(), name

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
if OUTER_LOSS == "classwise_top_loss":
    objective_label = f"top{OUTER_TOP_LOSS_PERCENT:g}pct_per_class"
elif OUTER_LOSS == "logsumexp":
    beta_label = str(OUTER_LSE_BETA).replace(".", "p")
    objective_label = f"logsumexp_beta_{beta_label}"
else:
    objective_label = OUTER_LOSS

drive_run = OUTPUT_ROOT / f"{objective_label}_all_sample_{stamp}"
drive_run.mkdir(parents=True, exist_ok=False)

for name in ["final_rater.pt", "log.txt", "summary.txt"]:
    shutil.copy2(local_exp / name, drive_run / name)
shutil.copytree(local_exp / "plots", drive_run / "plots")

print("Saved:", drive_run)
print("Rater:", drive_run / "final_rater.pt")
print("Plots:", drive_run / "plots")

In [ ]:
# 11. Summary + plot inventory
print((drive_run / "summary.txt").read_text(encoding="utf-8"))

plots = sorted((drive_run / "plots").rglob("*.png"))
print(f"Saved {len(plots)} PNG plots")
for p in plots:
    print(p.relative_to(drive_run))
print("Outer loss:", OUTER_LOSS)
print("Top-loss percentage (if selected):", OUTER_TOP_LOSS_PERCENT)
print("Log-sum-exp beta (if selected):", OUTER_LSE_BETA)
print("Outer objective uses group annotations:", OUTER_LOSS in {"worst_group_loss", "group_balanced_loss"})
